In [66]:
from dataclasses import dataclass
import json
import logging
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import sys

from modules import plotting
from modules import SequenceRepresentation as sr
from modules import training
from modules import utils

In [30]:
logging.basicConfig(format="%(asctime)s %(levelname)s: %(message)s", 
                    encoding='utf-8', level=logging.DEBUG)
#logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

In [37]:
#wd = Path("/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative")
wd = Path("/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321")
experiment_dirs = [f for f in wd.iterdir() if f.is_dir() and not (f.name.startswith("test") or f.name.startswith("slurm"))]
print(experiment_dirs[:max(3, len(experiment_dirs))])

[PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsSydhK562Gata2UcdUniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsSydhK562Elk112771IggrabUniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321

In [38]:
ed = experiment_dirs[0]

    genomes = sr.loadJSONGenomeList(str(ed / "test_sequences_0.json"))
    evaluator = training.loadMultiTrainingEvaluation(str(ed / "evaluator_test.json"), genomes)
    tr = evaluator.trainings[0]

    # better color scheme
    plotting.drawGeneLinks(tr.links, genomes[:20], "/mnt/c/Users/Matthis/Desktop/test.png", 
                        #kmerSites=[Links.Occurrence(genomes[0][0], i, '+') for i in [10, 50, 150]],
                        #maskingSites=[Links.Occurrence(genomes[0][0], i, '+') for i in [20, 40, 180]],
                        connectLinks=False,
                        genecols=["lightgray"]*20,
                        linkcols=["indigo"]*len(tr.links),
                        genewidth=20,
                        linkwidth=10,
                        sitecols={
                            'kmer sites': "#00ff007f",
                            'masking sites': "#00ff001a",
                            'stuff': 'white',
                            },
                        elementcols={
                            'peak_bed.tsv': "darkred",
                            'peak_fimo.tsv': "darkorange",
                            'peak_mast.tsv': "red",
                            'foo': 'black',
                            })

In [77]:
def load_mast(path: Path, k: int = None):
    """ Load MAST output from a best_hits file and return a DataFrame with the relevant columns 
    [sequence, hit_start, hit_end, hit_len, hit_center] as zero-based, end exclusive coordinates.
    Parameter k is optional to check that the length of the hits in the MAST output is consistent with k. """
    # if file is empty, return an empty DataFrame
    if path.stat().st_size == 0:
        return pd.DataFrame(columns=['sequence', 'hit_start', 'hit_end', 'hit_len', 'hit_center'])
    
    mast = pd.read_csv(str(path), sep="\s+",
                       names=["sequence", "(strand+/-)motif id", "alt_id", "-", "hit_start", "hit_end", "score", 
                               "hit_p-value"],
                       comment="#")
    # coordinates seem to be 1-based, and the end is inclusive (i.e. [start, end], not [start, end))
    # we want 0-based, end-exclusive coordinates, so we subtract 1 from the start and should be good
    mast['hit_start'] = mast['hit_start']-1
    assert (mast['hit_start'] >= 0).all(), str(mast['hit_start'].min())
    assert (mast['hit_end'] > mast['hit_start']).all(), f"{mast['hit_end'].min()} < {mast['hit_start'].min()}"
    mast['hit_len'] = mast['hit_end']-mast['hit_start']
    if k is not None:
        assert (mast['hit_len'] == k).all(), str(mast['hit_len'].value_counts())
    mast['hit_center'] = mast['hit_start']+(mast['hit_len']//2)

    return mast[['sequence', 'hit_start', 'hit_end', 'hit_len', 'hit_center']]


def load_fimo(path: Path, k: int = None):
    """ Load FIMO output from a best_site.narrowPeak file and return a DataFrame with the relevant columns
    [sequence, hit_start, hit_end, hit_len, hit_center].
    Parameter k is optional to check that the length of the hits in the FIMO output is consistent with k. """
    # if file is empty, return an empty DataFrame
    if path.stat().st_size == 0:
        return pd.DataFrame(columns=['sequence', 'hit_start', 'hit_end', 'hit_len', 'hit_center'])
    
    fimo = utils.readBEDFile(Path(str(path)))
    assert (fimo.columns == ['chrom', 'chromStart', 'chromEnd', 'name', 'score', 'strand', 
                            'signalValue', 'pValue', 'qValue', 'peak']).all(), str(fimo.columns)
    fimo.columns = ['sequence', 'hit_start', 'hit_end'] + fimo.columns[3:].tolist()
    assert (fimo['hit_start'] >= 0).all(), str(fimo['hit_start'].min())
    assert (fimo['hit_end'] > fimo['hit_start']).all(), f"{fimo['hit_end'].min()} < {fimo['hit_start'].min()}"
    assert (fimo['peak'] >= 0).all(), str(fimo['peak'].min())
    assert (fimo['peak'] < fimo['hit_end']).all(), fimo[['hit_end', 'peak']]
    fimo['hit_len'] = fimo['hit_end']-fimo['hit_start']
    if k is not None:
        assert (fimo['hit_len'] == k).all(), str(fimo['hit_len'].value_counts())
    fimo['hit_center'] = fimo['hit_start'] + fimo['peak']
    #assert (fimo['hit_center'] == (fimo['hit_start'] + (fimo['hit_len']//2))).all(), \
    #    fimo[['hit_center', 'hit_start', 'hit_len']]
    
    return fimo[['sequence', 'hit_start', 'hit_end', 'hit_len', 'hit_center']]

In [124]:
def evaluate_experiment(ed: Path, 
                        eval_file: str = "evaluator_test.json", 
                        neg_eval_file: str = "evaluator_negative_test.json", 
                        streme_eval_file: str = "STREME/streme_evaluator_dummymodel_test.json", 
                        streme_neg_eval_file: str = "STREME/streme_evaluator_dummymodel_negative_test.json",
                        fimo_sites_file: str = "fimo/best_site.narrowPeak",
                        fimo_neg_sites_file: str = "fimo/neg/best_site.narrowPeak",
                        fimo_streme_sites_file: str = "fimo/STREME/best_site.narrowPeak",
                        fimo_streme_neg_sites_file: str = "fimo/neg/STREME/best_site.narrowPeak",
                        mast_sites_file: str = "mast/best_hits.tsv",
                        mast_neg_sites_file: str = "mast/best_hits_neg.tsv",
                        mast_streme_sites_file: str = "mast/STREME/best_hits.tsv",
                        mast_streme_neg_sites_file: str = "mast/STREME/best_hits_neg.tsv",
                        training_seq_file: str = "training_sequences_0.json", 
                        test_seq_file: str = "test_sequences_0.json",
                        neg_test_seq_file: str = "negative_test_sequences_0.json",
                        settings_file: str = "settings.json",
                        show_plots: bool = True, silent: bool = False):
    assert ed.is_dir(), f"{ed} is not a directory"
    assert (ed / eval_file).is_file(), f"{ed / eval_file} does not exist"
    assert (ed / neg_eval_file).is_file(), f"{ed / neg_eval_file} does not exist"
    assert (ed / streme_eval_file).is_file(), f"{ed / streme_eval_file} does not exist"
    assert (ed / streme_neg_eval_file).is_file(), f"{ed / streme_neg_eval_file} does not exist"
    assert (ed / fimo_sites_file).is_file(), f"{ed / fimo_sites_file} does not exist"
    assert (ed / fimo_neg_sites_file).is_file(), f"{ed / fimo_neg_sites_file} does not exist"
    assert (ed / fimo_streme_sites_file).is_file(), f"{ed / fimo_streme_sites_file} does not exist"
    assert (ed / fimo_streme_neg_sites_file).is_file(), f"{ed / fimo_streme_neg_sites_file} does not exist"
    assert (ed / mast_sites_file).is_file(), f"{ed / mast_sites_file} does not exist"
    assert (ed / mast_neg_sites_file).is_file(), f"{ed / mast_neg_sites_file} does not exist"
    assert (ed / mast_streme_sites_file).is_file(), f"{ed / mast_streme_sites_file} does not exist"
    assert (ed / mast_streme_neg_sites_file).is_file(), f"{ed / mast_streme_neg_sites_file} does not exist"
    assert (ed / training_seq_file).is_file(), f"{ed / training_seq_file} does not exist"
    assert (ed / test_seq_file).is_file(), f"{ed / test_seq_file} does not exist"
    assert (ed / neg_test_seq_file).is_file(), f"{ed / neg_test_seq_file} does not exist"
    assert (ed / settings_file).is_file(), f"{ed / settings_file} does not exist"

    with open(ed / settings_file, "r") as f:
        settings = json.load(f)
    training_genomes = sr.loadJSONGenomeList(str(ed / training_seq_file))
    test_genomes = sr.loadJSONGenomeList(str(ed / test_seq_file))
    neg_test_genomes = sr.loadJSONGenomeList(str(ed / neg_test_seq_file))
    evaluator = training.loadMultiTrainingEvaluation(str(ed / eval_file), test_genomes)
    neg_evaluator = training.loadMultiTrainingEvaluation(str(ed / neg_eval_file), neg_test_genomes)
    streme_evaluator = training.loadMultiTrainingEvaluation(str(ed / streme_eval_file), test_genomes)
    streme_neg_evaluator = training.loadMultiTrainingEvaluation(str(ed / streme_neg_eval_file), neg_test_genomes)
    assert len(evaluator.trainings) == 1, f"expected 1 training, got {len(evaluator.trainings)}"
    assert len(neg_evaluator.trainings) == 1, f"expected 1 training, got {len(neg_evaluator.trainings)}"
    assert len(streme_evaluator.trainings) == 1, f"expected 1 training, got {len(streme_evaluator.trainings)}"
    assert len(streme_neg_evaluator.trainings) == 1, f"expected 1 training, got {len(streme_neg_evaluator.trainings)}"
    fimo_sites = load_fimo(ed / fimo_sites_file, settings['k'])
    fimo_neg_sites = load_fimo(ed / fimo_neg_sites_file, settings['k'])
    fimo_streme_sites = load_fimo(ed / fimo_streme_sites_file, settings['k'])
    fimo_streme_neg_sites = load_fimo(ed / fimo_streme_neg_sites_file, settings['k'])
    mast_sites = load_mast(ed / mast_sites_file, settings['k'])
    mast_neg_sites = load_mast(ed / mast_neg_sites_file, settings['k'])
    mast_streme_sites = load_mast(ed / mast_streme_sites_file, settings['k'])
    mast_streme_neg_sites = load_mast(ed / mast_streme_neg_sites_file, settings['k'])

    ref_types = set()
    for genomes in [training_genomes, test_genomes, neg_test_genomes]:
        for genome in genomes:
            for seq in genome:
                assert seq.elementsPossible(), f"no elements possible in {seq}"
                ref_types.update([e.type for e in seq.genomic_elements])

    def _evalHits(hits: list[training.Links.MultiLink], genomes: list[sr.Genome]):
        all_refs: dict[str, list[tuple[float, str]]] = {
            rt: [] for rt in ref_types
        }
        sidToIdcs: dict[str, tuple[int, int]] = {}
        for gid, genome in enumerate(genomes):
            for sid, seq in enumerate(genome):
                if seq.id not in sidToIdcs:
                    sidToIdcs[seq.id] = (gid, sid)
                elif not silent:
                    logging.warning(f"sequence id {seq.id} occurs multiple times")
                    
                assert seq.elementsPossible(), f"no elements in {seq}"
                for element in seq.genomic_elements:
                    ref, _ = element.getRelativePositions(seq)
                    rel_ref = 100*ref/len(seq)
                    all_refs[element.type].append((rel_ref, seq.id))

        ref_hit_distances: dict[str, list[tuple[int, tuple[int, str]]]] = {rt: [] for rt in ref_types}
        relative_hits: list[tuple[float, str]] = []
        for link in hits:
            for occs in link.occs:
                if len(occs) > 0:
                    assert occs[0].sequence.id in sidToIdcs, f"sequence id {occs[0].sequence.id} not found"
                    assert all([occs[0].sequence.id == o.sequence.id for o in occs]), \
                        "all occurrences must be in the same sequence"
                    
                    gid, sid = sidToIdcs[occs[0].sequence.id]
                    refs: dict[str, list[int]] = {rt: [] for rt in ref_types}
                    for element in genomes[gid].sequences[sid].genomic_elements:
                        ref, _ = element.getRelativePositions(genomes[gid][sid])
                        refs[element.type].append(ref)

                    for occ in occs:
                        hit_start = occ.position
                        hit_len = occ.sitelen
                        hit_end = hit_start + hit_len
                        hit_center = hit_start + hit_len // 2
                        relative_hits.append((100*hit_center/len(genomes[gid][sid]), occ.sequence.id))

                        for rt in refs.keys():
                            if len(refs[rt]) > 0:
                                ref_distances = []
                                for ref in refs[rt]:
                                    dist = 0 if hit_start <= ref < hit_end else min(abs(ref - hit_start), 
                                                                                    abs(ref - hit_end))
                                    ref_distances.append(dist)

                                i, min_d = min(enumerate(ref_distances), key=lambda x: x[1]) # argmin & min in one go
                                ref_hit_distances[rt].append((min_d, (refs[rt][i], occ.sequence.id))) # dist, (ref, seq)

        return all_refs, ref_hit_distances, relative_hits
    

    def _evalEvaluator(evaluator: training.MultiTrainingEvaluation, genomes: list[sr.Genome]):
        return _evalHits(evaluator.trainings[0].links, genomes)
    
    def _evalSites(sites: pd.DataFrame, genomes: list[sr.Genome]):
        assert sites.columns.tolist() == ['sequence', 'hit_start', 'hit_end', 'hit_len', 'hit_center'], \
            f"unexpected columns {sites.columns.tolist()}"
        # create multilinks from sites
        seqidToIdcs = {}
        for gid, genome in enumerate(genomes):
            for sid, seq in enumerate(genome):
                if seq.species not in seqidToIdcs:
                    seqidToIdcs[seq.species] = (gid, sid) # seq.species should match the sequences in DataFrame
                elif not silent:
                    #print(seq.toDict())
                    logging.warning(f"sequence id {seq.chromosome} occurs multiple times")
                    #assert False
                
        assert all([sid in seqidToIdcs for sid in sites['sequence']]), \
            f"{len([s for s in sites['sequence'] if s not in seqidToIdcs])}/{len(set(sites['sequence']))} sequence ids {sorted(set([s for s in sites['sequence'] if s not in seqidToIdcs]))}\n\n" \
                + f"not found in {len(seqidToIdcs.keys())} genomes {sorted(seqidToIdcs.keys())}"
        occs = [training.Links.Occurrence(sequence=genomes[seqidToIdcs[seqid][0]][seqidToIdcs[seqid][1]], 
                                          position=hit_start, 
                                          strand="+", # ignoring strand information for now
                                          sitelen=hit_len) \
                for seqid, hit_start, hit_end, hit_len, hit_center in sites.itertuples(index=False)]
        links = [training.Links.MultiLink(occs, singleProfile=True)] # assume this, must be changed if we want to distinguish single profiles
        return _evalHits(links, genomes)

    # get evaluation results
    all_refs, ref_hit_distances, relative_hits = _evalEvaluator(evaluator, test_genomes)
    _, ref_hit_distances_streme, relative_hits_streme = _evalEvaluator(streme_evaluator, test_genomes)
    _, ref_hit_distances_fimo, relative_hits_fimo = _evalSites(fimo_sites, test_genomes)
    _, ref_hit_distances_fimo_streme, relative_hits_fimo_streme = _evalSites(fimo_streme_sites, test_genomes)
    _, ref_hit_distances_mast, relative_hits_mast = _evalSites(mast_sites, test_genomes)
    _, ref_hit_distances_mast_streme, relative_hits_mast_streme = _evalSites(mast_streme_sites, test_genomes)

    all_refs_neg, ref_hit_distances_neg, relative_hits_neg = _evalEvaluator(neg_evaluator, neg_test_genomes)
    assert all([len(all_refs_neg[rt]) == 0 for rt in all_refs_neg.keys()]), \
        "no reference sites should be found in negative test sequences"
    assert all([len(ref_hit_distances_neg[rt]) == 0 for rt in ref_hit_distances_neg.keys()]), \
        "no reference site - hit distances should be found in negative test sequences"
    _, ref_hit_distances_neg_streme, relative_hits_neg_streme = _evalEvaluator(streme_neg_evaluator, neg_test_genomes)
    assert all([len(ref_hit_distances_neg_streme[rt]) == 0 for rt in ref_hit_distances_neg_streme.keys()]), \
        "no reference site - hit distances should be found in negative test sequences"
    _, ref_hit_distances_fimo_neg, relative_hits_fimo_neg = _evalSites(fimo_neg_sites, neg_test_genomes)
    _, ref_hit_distances_fimo_streme_neg, relative_hits_fimo_streme_neg = _evalSites(fimo_streme_neg_sites, 
                                                                                     neg_test_genomes)
    _, ref_hit_distances_mast_neg, relative_hits_mast_neg = _evalSites(mast_neg_sites, neg_test_genomes)
    _, ref_hit_distances_mast_streme_neg, relative_hits_mast_streme_neg = _evalSites(mast_streme_neg_sites, 
                                                                                     neg_test_genomes)
    assert all([len(ref_hit_distances_fimo_neg[rt]) == 0 for rt in ref_hit_distances_fimo_neg.keys()]), \
        "no reference site - hit distances should be found in negative test sequences"
    assert all([len(ref_hit_distances_fimo_streme_neg[rt]) == 0 for rt in ref_hit_distances_fimo_streme_neg.keys()]), \
        "no reference site - hit distances should be found in negative test sequences"
    assert all([len(ref_hit_distances_mast_neg[rt]) == 0 for rt in ref_hit_distances_mast_neg.keys()]), \
        "no reference site - hit distances should be found in negative test sequences"
    assert all([len(ref_hit_distances_mast_streme_neg[rt]) == 0 for rt in ref_hit_distances_mast_streme_neg.keys()]), \
        "no reference site - hit distances should be found in negative test sequences"

    if show_plots:
        fig = plotting.ownPlotlyHist({rt: [t[0] for t in all_refs[rt]] for rt in all_refs})
        fig.update_layout(title="Reference site distribution in test sequences", 
                          xaxis_title="relative position * 100", yaxis_title="reference site count")
        fig.show()
        # # sanitiy check histogram function
        # plt.figure(figsize=(16,9))
        # plt.hist([t[0] for t in all_refs['peak_fimo.tsv']], bins=range(0,101,1), edgecolor='black', density=True)
        # plt.title("FIMO reference site distribution in test sequences")
        # plt.xlabel("relative position * 100")
        # plt.ylabel("reference site count")
        # plt.show()

        fig = plotting.ownPlotlyHist({f"{sites} {rt}": [t[0] for t in dists[rt]] \
                                        for sites, dists in {'ProfileFinding': ref_hit_distances, 
                                                             'FIMO': ref_hit_distances_fimo, 
                                                             'MAST': ref_hit_distances_mast}.items() \
                                            for rt in dists}, 
                                     rel=True)
        fig.update_layout(title="Distance of hits to reference sites in test sequences", 
                          xaxis_title="distance to closest reference site", yaxis_title="relative frequency")
        fig.show()
        # # sanitiy check histogram function
        # plt.figure(figsize=(16,9))
        # plt.hist([t[0] for t in ref_hit_distances['peak_fimo.tsv']], bins=range(0,max([t[0] for t in ref_hit_distances['peak_fimo.tsv']])+1,2), edgecolor='black', density=True)
        # plt.title("Distance of hits to FIMO reference sites in test sequences ")
        # plt.xlabel("distance to closest reference site")
        # plt.ylabel("relative frequency")
        # plt.show()

        fig = plotting.ownPlotlyHist({f"{sites} {rt}": [t[0] for t in dists[rt]] \
                                        for sites, dists in {'ProfileFinding STREME': ref_hit_distances_streme, 
                                                             'FIMO STREME': ref_hit_distances_fimo_streme, 
                                                             'MAST STREME': ref_hit_distances_mast_streme}.items() \
                                            for rt in dists}, 
                                     rel=True)
        fig.update_layout(title="Distance of hits to reference sites in test sequences | STREME", 
                          xaxis_title="distance to closest reference site", yaxis_title="relative frequency")
        fig.show()

        fig = plotting.ownPlotlyHist({f"relative hits {mode}": [t[0] for t in hits] \
                                      for mode, hits in {'ProfileFinding': relative_hits,
                                                         'FIMO': relative_hits_fimo,
                                                         'MAST': relative_hits_mast}.items()})
        fig.update_layout(title="Relative hit positions in test sequences", 
                          xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()
        # # sanitiy check histogram function
        # plt.figure(figsize=(16,9))
        # plt.hist([t[0] for t in relative_hits], bins=range(0,101,1), edgecolor='black', density=True)
        # plt.title("Relative hit positions in test sequences")
        # plt.xlabel("relative position * 100")
        # plt.ylabel("hit count")
        # plt.show()

        fig = plotting.ownPlotlyHist({f"relative hits {mode}": [t[0] for t in hits] \
                                      for mode, hits in {'ProfileFinding STREME': relative_hits_streme,
                                                         'FIMO STREME': relative_hits_fimo_streme,
                                                         'MAST STREME': relative_hits_mast_streme}.items()})
        fig.update_layout(title="Relative hit positions in test sequences | STREME", 
                          xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()

        fig = plotting.ownPlotlyHist({f"relative hits {mode}": [t[0] for t in hits] \
                                      for mode, hits in {'ProfileFinding negative': relative_hits_neg,
                                                         'FIMO negative': relative_hits_fimo_neg,
                                                         'MAST negative': relative_hits_mast_neg}.items()})
        fig.update_layout(title="Relative hit positions in negative sequences", 
                          xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()

        fig = plotting.ownPlotlyHist({f"relative hits {mode}": [t[0] for t in hits] \
                                      for mode, hits in {'ProfileFinding negative STREME': relative_hits_neg_streme,
                                                         'FIMO negative STREME': relative_hits_fimo_streme_neg,
                                                         'MAST negative STREME': relative_hits_mast_streme_neg}.items()}
                                                         )
        fig.update_layout(title="Relative hit positions in negative sequences | STREME", 
                          xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()

    

    # if show_plots:
    #     fig = plotting.ownPlotlyHist({"relative hits": [t[0] for t in relative_hits_neg]})
    #     fig.update_layout(title="Relative hit positions in negative test sequences", 
    #                     xaxis_title="relative position * 100", yaxis_title="hit count")
    #     fig.show()

    # # --- Streme ---
    
    # if show_plots:
    #     fig = plotting.ownPlotlyHist({rt: [t[0] for t in ref_hit_distances_streme[rt]] \
    #                                         for rt in ref_hit_distances_streme}, 
    #                                  rel=True)
    #     fig.update_layout(title="Distance of hits to reference sites in test sequences | STREME", 
    #                     xaxis_title="distance to closest reference site", yaxis_title="relative frequency")
    #     fig.show()

    #     fig = plotting.ownPlotlyHist({"relative hits": [t[0] for t in relative_hits_streme]})
    #     fig.update_layout(title="Relative hit positions in test sequences | STREME", 
    #                     xaxis_title="relative position * 100", yaxis_title="hit count")
    #     fig.show()

    
    # if show_plots:
    #     fig = plotting.ownPlotlyHist({"relative hits": [t[0] for t in relative_hits_neg_streme]})
    #     fig.update_layout(title="Relative hit positions in negative test sequences | STREME", 
    #                     xaxis_title="relative position * 100", yaxis_title="hit count")
    #     fig.show()

    @dataclass
    class EvalResult:
        genomes: list[sr.Genome]
        neg_genomes: list[sr.Genome]
        all_refs: dict[str, list[tuple[float, str]]]
        ref_hit_distances: dict[str, list[tuple[int, tuple[int, str]]]]
        relative_hits: list[tuple[float, str]]
        relative_hits_neg: list[tuple[float, str]]
        ref_hit_distances_streme: dict[str, list[tuple[int, tuple[int, str]]]]
        relative_hits_streme: list[tuple[float, str]]
        relative_hits_neg_streme: list[tuple[float, str]]

    return EvalResult(test_genomes, neg_test_genomes, 
                      all_refs, ref_hit_distances, relative_hits, relative_hits_neg,
                      ref_hit_distances_streme, relative_hits_streme, relative_hits_neg_streme)

# _ = evaluate_experiment(ed)

In [125]:
print(experiment_dirs[0])
evaluate_experiment(experiment_dirs[0])

/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak


AssertionError: 1804/2076 sequence ids ['chr10:100248891-100249091', 'chr10:103070918-103071118', 'chr10:103113664-103113864', 'chr10:113943469-113943589', 'chr10:11940640-11940840', 'chr10:12238065-12238265', 'chr10:124713640-124713840', 'chr10:124895390-124895538', 'chr10:131909091-131909291', 'chr10:17243610-17243810', 'chr10:17428792-17428992', 'chr10:18940442-18940642', 'chr10:21330663-21330863', 'chr10:28821615-28821815', 'chr10:33247268-33247468', 'chr10:35415745-35415945', 'chr10:38149902-38150102', 'chr10:43603841-43604041', 'chr10:43903216-43903416', 'chr10:43904614-43904814', 'chr10:45474235-45474435', 'chr10:45966111-45966311', 'chr10:46090263-46090463', 'chr10:49732379-49732579', 'chr10:50369675-50369875', 'chr10:53459313-53459513', 'chr10:5727241-5727441', 'chr10:63511859-63512059', 'chr10:64576079-64576308', 'chr10:64603440-64603640', 'chr10:65028859-65029059', 'chr10:69609158-69609358', 'chr10:69644305-69644505', 'chr10:70480801-70481001', 'chr10:7070246-7070446', 'chr1

In [ ]:
def evaluate_multiple_experiments(experiment_dirs: list[Path], show_plots: bool = True):
    sequences: set[str] = set()
    sequences_with_hits: set[str] = set()
    sequences_with_hits_streme: set[str] = set()
    matches: list[float] = []
    matches_streme: list[float] = []
    sequences_with_refsites: dict[str, set[str]] = {}
    sequences_with_hits_on_refsites: dict[str, set[str]] = {}
    sequences_with_hits_on_refsites_streme: dict[str, set[str]] = {}
    refsites: dict[str, list[float]] = {}
    hits_on_refsites: dict[str, list[tuple[int, str]]] = {}
    hits_on_refsites_streme: dict[str, list[tuple[int, str]]] = {}
    ref_distances: dict[str, list[int]] = {}
    ref_distances_streme: dict[str, list[int]] = {}
    negative_sequences: set[str] = set()
    negative_sequences_with_hits: set[str] = set()
    negative_sequences_with_hits_streme: set[str] = set()
    negative_matches: list[float] = []
    negative_matches_streme: list[float] = []

    for ed in experiment_dirs:
        try:
            result = evaluate_experiment(ed, show_plots=False, silent=True)
        except Exception as e:
            print(f"Could not evaluate experiment {ed}:\n\t{e}")
            continue

        sequences.update([seq.id for genome in result.genomes for seq in genome])
        negative_sequences.update([seq.id for genome in result.neg_genomes for seq in genome])
        for rt in result.all_refs.keys():
            if rt not in refsites:
                refsites[rt] = []
            if rt not in sequences_with_refsites:
                sequences_with_refsites[rt] = set()

            refsites[rt].extend([t[0] for t in result.all_refs[rt]])
            sequences_with_refsites[rt].update([t[1] for t in result.all_refs[rt]])

        for rt in result.ref_hit_distances:
            if rt not in hits_on_refsites:
                hits_on_refsites[rt] = []
            if rt not in sequences_with_hits_on_refsites:
                sequences_with_hits_on_refsites[rt] = set()
            if rt not in ref_distances:
                ref_distances[rt] = []

            ref_distances[rt].extend([t[0] for t in result.ref_hit_distances[rt]])
            refhits = [t[1] for t in result.ref_hit_distances[rt] if t[0] == 0]
            hits_on_refsites[rt].extend(refhits)
            sequences_with_hits_on_refsites[rt].update([t[1] for t in refhits])

        for rt in result.ref_hit_distances_streme:
            if rt not in hits_on_refsites_streme:
                hits_on_refsites_streme[rt] = []
            if rt not in sequences_with_hits_on_refsites_streme:
                sequences_with_hits_on_refsites_streme[rt] = set()
            if rt not in ref_distances_streme:
                ref_distances_streme[rt] = []

            ref_distances_streme[rt].extend([t[0] for t in result.ref_hit_distances_streme[rt]])
            refhits = [t[1] for t in result.ref_hit_distances_streme[rt] if t[0] == 0]
            hits_on_refsites_streme[rt].extend(refhits)
            sequences_with_hits_on_refsites_streme[rt].update([t[1] for t in refhits])

        for relpos, seqid in result.relative_hits:
            sequences_with_hits.add(seqid)
            matches.append(relpos)

        for relpos, seqid in result.relative_hits_neg:
            negative_sequences_with_hits.add(seqid)
            negative_matches.append(relpos)

        for relpos, seqid in result.relative_hits_streme:
            sequences_with_hits_streme.add(seqid)
            matches_streme.append(relpos)

        for relpos, seqid in result.relative_hits_neg_streme:
            negative_sequences_with_hits_streme.add(seqid)
            negative_matches_streme.append(relpos)



    print(f"""
Number of test sequences: {len(sequences)}
Number of test sequences with reference sites: {[str(rt)+' '+str(len(sequences_with_refsites[rt])) for rt in sequences_with_refsites]}
Number of neg. sequences: {len(negative_sequences)}

---

Number of hits (test): {len(matches)} | {len(matches)/len(sequences):.2f} hits per seq
Number of hits (neg.): {len(negative_matches)} | {len(negative_matches)/len(negative_sequences):.2f} hits per seq

Number of hits STREME (test): {len(matches_streme)} | {len(matches_streme)/len(sequences):.2f} hits per seq
Number of hits STREME (neg.): {len(negative_matches_streme)} | {len(negative_matches_streme)/len(negative_sequences):.2f} hits per seq

---

Test sequences with >= 1 hit: {100*len(sequences_with_hits)/len(sequences):.2f}% ({len(sequences_with_hits)})
Neg. sequences with >= 1 hit: {100*len(negative_sequences_with_hits)/len(negative_sequences):.2f}% ({len(negative_sequences_with_hits)})

Test sequences with >= 1 hit STREME: {100*len(sequences_with_hits_streme)/len(sequences):.2f}% ({len(sequences_with_hits_streme)})
Neg. sequences with >= 1 hit STREME: {100*len(negative_sequences_with_hits_streme)/len(negative_sequences):.2f}% ({len(negative_sequences_with_hits_streme)})

---""")
    
    for rt in refsites:
        # avoid key errors, although unlikely to happen
        for v in [sequences_with_refsites, sequences_with_hits_on_refsites, sequences_with_hits_on_refsites_streme]:
            if rt not in v:
                v[rt] = set()
        for v in [hits_on_refsites, hits_on_refsites_streme]:
            if rt not in v:
                v[rt] = []

        print(f"""
{rt} reference sites:
\tNumber of reference sites: {len(refsites[rt])}
\tSequences with reference sites: {100*len(sequences_with_refsites[rt])/len(sequences):.2f}% ({len(sequences_with_refsites[rt])})
\tReference sites per sequence: {len(refsites[rt])/len(sequences):.2f} (all) | {len(refsites[rt])/len(sequences_with_refsites[rt]):.2f} (seq. w/ ref. sites)

\tSequences with hits on reference sites: {100*len(sequences_with_hits_on_refsites[rt])/len(sequences):.2f}% ({len(sequences_with_hits_on_refsites[rt])})
\tReference sites hit: {100*len(set(hits_on_refsites[rt]))/len(refsites[rt]):.2f}% ({len(set(hits_on_refsites[rt]))})

\tSequences with hits on reference sites STREME: {100*len(sequences_with_hits_on_refsites_streme[rt])/len(sequences):.2f}% ({len(sequences_with_hits_on_refsites_streme[rt])})
\tReference sites hit STREME: {100*len(set(hits_on_refsites_streme[rt]))/len(refsites[rt]):.2f}% ({len(set(hits_on_refsites_streme[rt]))})
""")
    
    if show_plots:
        fig = plotting.ownPlotlyHist(refsites)
        fig.update_layout(title="Reference site distribution in test sequences", 
                        xaxis_title="relative position * 100", yaxis_title="site count")
        fig.show()
        # # # sanitiy check histogram function
        # plt.figure(figsize=(16,9))
        # plt.hist(refsites['peak_fimo.tsv'], bins=range(0,101,1), edgecolor='black', density=True)
        # plt.title("FIMO reference site distribution in test sequences")
        # plt.xlabel("relative position * 100")
        # plt.ylabel("site count")
        # plt.show()

        fig = plotting.ownPlotlyHist(ref_distances, rel=True)
        fig.update_layout(title="Distance of hits to reference sites in test sequences", 
                          xaxis_title="distance to closest reference site", yaxis_title="relative frequency")
        fig.show()
        # # sanitiy check histogram function
        # plt.figure(figsize=(16,9))
        # plt.hist(ref_distances['peak_fimo.tsv'], bins=range(0,max(ref_distances['peak_fimo.tsv'])+1,2), edgecolor='black', density=True)
        # plt.title("Distance of hits to FIMO reference sites in test sequences ")
        # plt.xlabel("distance to closest reference site")
        # plt.ylabel("relative frequency")
        # plt.show()

        fig = plotting.ownPlotlyHist({"relative hits": matches})
        fig.update_layout(title="Relative hit positions in test sequences", 
                          xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()
        # # sanitiy check histogram function
        # plt.figure(figsize=(16,9))
        # plt.hist(matches, bins=range(0,101,1), edgecolor='black', density=True)
        # plt.title("Relative hit positions in test sequences")
        # plt.xlabel("relative position * 100")
        # plt.ylabel("hit count")
        # plt.show()

        fig = plotting.ownPlotlyHist({"relative hits": negative_matches})
        fig.update_layout(title="Relative hit positions in negative test sequences", 
                          xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()

    # --- Streme ---
        fig = plotting.ownPlotlyHist(ref_distances_streme, rel=True)
        fig.update_layout(title="Distance of hits to reference sites in test sequences | STREME", 
                          xaxis_title="distance to closest reference site", yaxis_title="relative frequency")
        fig.show()

        fig = plotting.ownPlotlyHist({"relative hits": matches_streme})
        fig.update_layout(title="Relative hit positions in test sequences | STREME", 
                        xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()

        fig = plotting.ownPlotlyHist({"relative hits": negative_matches_streme})
        fig.update_layout(title="Relative hit positions in negative test sequences | STREME", 
                          xaxis_title="relative position * 100", yaxis_title="hit count")
        fig.show()

In [92]:
evaluate_multiple_experiments(experiment_dirs)

2025-03-24 19:50:19,678 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)
2025-03-24 19:50:19,852 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)
2025-03-24 19:50:19,931 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)
2025-03-24 19:50:20,018 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)
2025-03-24 19:50:21,818 WARNING: sequence id chr10:112

Could not evaluate experiment /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak:
	/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
Could not evaluate experiment /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak:
	/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
Could not evaluate experiment /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak:
	/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsUtaK562CtcfUniPk.n

2025-03-24 19:50:30,524 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:72510161-72511226:0:0-1,065 (2x), chr6:27655855-27656374:0:0-519 (2x)
2025-03-24 19:50:30,637 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr17:72510161-72511226:0:0-1,065 (2x), negative_chr6:27655855-27656374:0:0-519 (2x)
2025-03-24 19:50:30,724 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:72510161-72511226:0:0-1,065 (2x), chr6:27655855-27656374:0:0-519 (2x)
2025-03-24 19:50:30,765 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr17:72510161-72511226:0:0-1,065 (2x), negative_chr6:27655855-27656374:0:0-519 (2x)
2025-03-24 19:50:32,104 WARNING: sequence id chr6:27655855-27656374:0:0-519 occurs multiple times
2025-03-24 19:50:32,107 WARNING: sequence id chr17:72510161-72511226:0:0-1,065 occurs multiple times
2025-03-24 19:50:

Could not evaluate experiment /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak:
	/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-24 19:51:23,345 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr6:11537625-11538146:0:0-521 (2x), chr2:175200793-175202077:0:0-1,284 (2x), chrX:37544927-37545500:0:0-573 (2x), chr1:36786772-36787466:0:0-694 (2x), chr1:16173852-16174501:0:0-649 (2x), chr1:234735500-234736178:0:0-678 (2x), chr3:38388039-38388628:0:0-589 (2x), chr19:11071274-11071738:0:0-464 (2x), chr12:124086253-124086801:0:0-548 (2x), chr3:193788616-193789324:0:0-708 (2x), chr6:144536966-144537517:0:0-551 (2x), chr6:159065362-159065766:0:0-404 (2x), chr1:112281896-112282425:0:0-529 (2x), chr7:129074035-129074584:0:0-549 (2x), chr8:98787854-98788312:0:0-458 (2x), chr5:139027661-139028005:0:0-344 (2x), chr1:33116408-33117190:0:0-782 (2x), chr12:108908671-108909083:0:0-412 (2x), chr4:90032070-90032874:0:0-804 (2x), chr7:151328925-151329711:0:0-786 (2x), chr15:41952213-41953334:0:0-1,121 (2x), chr14:100659117-100659576:0:0-459 (2x), chr15:41055323-41056071:0:0-748 (2x), chr1

...

Could not evaluate experiment /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsSydhK562CebpbIggrabUniPk.narrowPeak:
	/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsSydhK562CebpbIggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-24 19:52:36,698 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr9:132597612-132598067:0:0-455 (2x), chr8:142427409-142428437:0:0-1,028 (2x), chr19:19516712-19517481:0:0-769 (2x), chr12:42631191-42631813:0:0-622 (2x), chr7:65447017-65447496:0:0-479 (2x), chr11:72853093-72853622:0:0-529 (2x), chr7:154793980-154794915:0:0-935 (2x), chr5:96270680-96271129:0:0-449 (2x), chr1:156662686-156663301:0:0-615 (2x), chr20:62526719-62527111:0:0-392 (2x), chr3:13036359-13036852:0:0-493 (2x), chr7:43798050-43798484:0:0-434 (2x), chr20:49307836-49308441:0:0-605 (2x), chr16:83986480-83986895:0:0-415 (2x), chr4:6784832-6785381:0:0-549 (2x), chr19:10713038-10713640:0:0-602 (2x), chr22:20849578-20850430:0:0-852 (2x), chr2:106014802-106015752:0:0-950 (2x), chr3:23847521-23848789:0:0-1,268 (2x), chr9:131418639-131419158:0:0-519 (2x), chr19:4867014-4867811:0:0-797 (2x), chr1:17764060-17764595:0:0-535 (2x), chr22:19701615-19702717:0:0-1,102 (2x), chr16:85415487

Could not evaluate experiment /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsUwK562CtcfUniPk.narrowPeak:
	/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250321/wgEncodeAwgTfbsUwK562CtcfUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-24 19:53:24,808 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr4:128702821-128703160:0:0-339 (2x), chr7:100136668-100137233:0:0-565 (2x), chr22:19705322-19706321:0:0-999 (2x), chr1:197871671-197872361:0:0-690 (2x), chr1:43123582-43124140:0:0-558 (2x), chr17:61926089-61927170:0:0-1,081 (2x), chr2:106015376-106015966:0:0-590 (2x), chr22:20067214-20068089:0:0-875 (2x), chr9:37485657-37486115:0:0-458 (2x), chr1:155022843-155023312:0:0-469 (2x), chr11:118868258-118868750:0:0-492 (2x), chr7:135346967-135347443:0:0-476 (2x), chr11:67764043-67764415:0:0-372 (2x), chr7:148725666-148726250:0:0-584 (2x), chr19:12917200-12917697:0:0-497 (2x), chr16:89939464-89940035:0:0-571 (2x), chr13:115079619-115080065:0:0-446 (2x), chr16:81129904-81130331:0:0-427 (2x), chr11:125495140-125495604:0:0-464 (2x), chr1:12123170-12123898:0:0-728 (2x), chr19:59025184-59025713:0:0-529 (2x), chr2:232328405-232328765:0:0-360 (2x), chr5:43120774-43121302:0:0-528 (2x), chr


Number of test sequences: 131898
Number of test sequences with reference sites: ['peak_fimo.tsv 33433', 'peak_mast.tsv 32046', 'peak_bed.tsv 131883']
Number of neg. sequences: 131898

---

Number of hits (test): 193229 | 1.46 hits per seq
Number of hits (neg.): 50144 | 0.38 hits per seq

Number of hits STREME (test): 36023 | 0.27 hits per seq
Number of hits STREME (neg.): 3561 | 0.03 hits per seq

---

Test sequences with >= 1 hit: 59.79% (78862)
Neg. sequences with >= 1 hit: 28.13% (37109)

Test sequences with >= 1 hit STREME: 19.62% (25882)
Neg. sequences with >= 1 hit STREME: 2.36% (3118)

---

peak_fimo.tsv reference sites:
	Number of reference sites: 34213
	Sequences with reference sites: 25.35% (33433)
	Reference sites per sequence: 0.26 (all) | 1.02 (seq. w/ ref. sites)

	Sequences with hits on reference sites: 4.85% (6400)
	Reference sites hit: 18.71% (6400)

	Sequences with hits on reference sites STREME: 5.43% (7157)
	Reference sites hit STREME: 20.92% (7157)


peak_mast.tsv

---

In [16]:
def evaluate(experiment_dirs, evaluator_path, neg_evaluator_path = None):
    n_test_seqs = 0 # track total number of test seqs over all experiments to compare between runs, even if single experiments fail
    n_peaks = {}
    n_skipped_seqs = 0
    n_skipped_peaks = {}
    skipped_experiments = []
    glob_seqs = {}
    glob_seqs_neg = {}
    for ed in experiment_dirs:
        assert (ed / 'test_sequences_0.json').exists()
        skipping = (not (ed / evaluator_path).exists()) or (neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists())
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
        n_test_seqs += sum([len(g) for g in testdata])
        if skipping:
            n_skipped_seqs += sum([len(g) for g in testdata])
            skipped_experiments.append(str(ed))

        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    peaksrc = e.source
                    if peaksrc not in n_peaks:
                        n_peaks[peaksrc] = 0
                    n_peaks[peaksrc] += 1
                    if skipping:
                        if peaksrc not in n_skipped_peaks:
                            n_skipped_peaks[peaksrc] = 0
                        n_skipped_peaks[peaksrc] += 1

        if not (ed / evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / evaluator_path} does not exist")
            continue

        if neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / neg_evaluator_path} does not exist")
            continue
        
        # retrieve peaks from test data (peaks are stored as genomic elements in the sequences)
        peaks = {}
        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    peaksrc = e.source
                    if peaksrc not in peaks:
                        peaks[peaksrc] = {}
                    if s.id not in peaks[peaksrc]:
                        peaks[peaksrc][s.id] = []
                    peakstart, peakend = e.getRelativePositions(s)
                    assert peakend == peakstart + 1, f"Peak {e} is not a single base pair"
                    peaks[peaksrc][s.id].append(peakstart)

        # store how many times and where each sequence was hit
        seqdict = {s.id: [] for g in testdata for s in g} 
        evaluator = training.loadMultiTrainingEvaluation(str(ed / evaluator_path), testdata)
        assert len(evaluator.trainings) == 1
        tr = evaluator.trainings[0]
        for link in tr.links:
            for occs in link.occs: # list of list of occurrences
                for occ in occs:
                    assert occ.sequence.id in seqdict
                    seqdict[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        if neg_evaluator_path is not None:
            # store how many times and where each sequence was hit
            assert (ed / 'negative_test_sequences_0.json').exists()
            testdata_neg = sr.loadJSONGenomeList(str(ed / 'negative_test_sequences_0.json'))
            seqdict_neg = {s.id: [] for g in testdata_neg for s in g} 
            evaluator_neg = training.loadMultiTrainingEvaluation(str(ed / neg_evaluator_path), testdata_neg)
            assert len(evaluator_neg.trainings) == 1
            tr = evaluator_neg.trainings[0]
            for link in tr.links:
                for occs in link.occs: # list of list of occurrences
                    for occ in occs:
                        assert occ.sequence.id in seqdict_neg
                        seqdict_neg[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        # globally count how many times each sequence was hit
        for sid in seqdict:
            if sid not in glob_seqs:
                glob_seqs[sid] = {'hits': 0, 'peaks': {}}
            glob_seqs[sid]['hits'] += len(seqdict[sid])
            for peaksrc in peaks:
                if sid in peaks[peaksrc]:
                    if peaksrc not in glob_seqs[sid]['peaks']:
                        glob_seqs[sid]['peaks'][peaksrc] = {'peaks': set(), 'hits': set()}
                    glob_seqs[sid]['peaks'][peaksrc]['peaks'].update(peaks[peaksrc][sid])
                    for p in peaks[peaksrc][sid]:
                        for hit in seqdict[sid]:
                            if hit[0] <= p < hit[1]:
                                glob_seqs[sid]['peaks'][peaksrc]['hits'].add(p)

        if neg_evaluator_path is not None:
            for sid in seqdict_neg:
                if sid not in glob_seqs_neg:
                    glob_seqs_neg[sid] = {'hits': 0}
                glob_seqs_neg[sid]['hits'] += len(seqdict_neg[sid])

    nseqs = len(glob_seqs.keys())
    nseqs_hit = len([k for k in glob_seqs.keys() if glob_seqs[k]['hits'] > 0])
    nmatches = sum([v['hits'] for v in glob_seqs.values()])

    print(f"Total number of test sequences: {n_test_seqs} | Number of peaks in these sequences: {n_peaks}")
    print(f"Skipped number of test sequences: {n_skipped_seqs} | Number of peaks in these sequences: {n_skipped_peaks}")
    print(f"Skipped {len(skipped_experiments)}/{len(experiment_dirs)} experiments: {skipped_experiments}")
    print(f"Number of sequences: {nseqs}")
    print(f"Number of sequences with hits: {nseqs_hit} | ratio: {nseqs_hit/nseqs:.2f}")
    print(f"Number of matches: {nmatches} | ratio: {nmatches/nseqs:.2f}")
    print()

    for peaksrc in peaks:
        print(f"Peak source: {peaksrc}")
        n_peak_seqs = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks']])
        n_peak_seqs_with_hits = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks'] and glob_seqs[k]['peaks'][peaksrc]['hits']])
        n_peaks = sum([len(v) for v in peaks[peaksrc].values()])
        n_peaks_hit = sum([len(v['peaks'][peaksrc]['hits']) for v in glob_seqs.values() if peaksrc in v['peaks']])

        print(f"Number of sequences with peaks: {n_peak_seqs}")
        print(f"Number of sequences with hits on peaks: {n_peak_seqs_with_hits} | ratio: {n_peak_seqs_with_hits/n_peak_seqs:.2f}")
        print(f"Number of peaks: {n_peaks}")
        print(f"Number of hits on peaks: {n_peaks_hit} | ratio: {n_peaks_hit/n_peaks:.2f}")
        print()

    if neg_evaluator_path is not None:
        nseqs_neg = len(glob_seqs_neg.keys())
        nseqs_hit_neg = len([k for k in glob_seqs_neg.keys() if glob_seqs_neg[k]['hits'] > 0])
        nmatches_neg = sum([v['hits'] for v in glob_seqs_neg.values()])

        print(f"Number of negative sequences: {nseqs_neg}")
        print(f"Number of negative sequences with hits: {nseqs_hit_neg} | ratio: {nseqs_hit_neg/nseqs_neg:.2f}")
        print(f"Number of negative matches: {nmatches_neg} | ratio: {nmatches_neg/nseqs_neg:.2f}")
        print()

In [17]:
evaluate(experiment_dirs, 'evaluator_test.json', 'evaluator_negative_test.json')

2025-03-21 11:15:04,509 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)
2025-03-21 11:15:06,575 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)
2025-03-21 11:15:17,050 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:42295995-42296645:0:0-650 (2x), chr2:145089806-145090327:0:0-521 (2x), chr3:42641896-42642557:0:0-661 (2x)
2025-03-21 11:15:18,748 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr17:42295995-42296645:0:0-650 (2x), negative_chr2:145089806-145090327:0:0-521 (2x), negative_chr3:42641896-42642557:0:0-661 (2x)
2025-03-21 11:15:32,398 WARNING: [loadMultiTrainingE

Total number of test sequences: 216125 | Number of peaks in these sequences: {'bed.tsv': 217928, 'fimo.tsv': 85725, 'mast.tsv': 93968}
Skipped number of test sequences: 0 | Number of peaks in these sequences: {}
Skipped 0/40 experiments: []
Number of sequences: 215834
Number of sequences with hits: 121702 | ratio: 0.56
Number of matches: 284647 | ratio: 1.32

Peak source: bed.tsv
Number of sequences with peaks: 215808
Number of sequences with hits on peaks: 20487 | ratio: 0.09
Number of peaks: 941
Number of hits on peaks: 20494 | ratio: 21.78

Peak source: fimo.tsv
Number of sequences with peaks: 84777
Number of sequences with hits on peaks: 28794 | ratio: 0.34
Number of peaks: 488
Number of hits on peaks: 28794 | ratio: 59.00

Peak source: mast.tsv
Number of sequences with peaks: 81851
Number of sequences with hits on peaks: 29450 | ratio: 0.36
Number of peaks: 735
Number of hits on peaks: 31378 | ratio: 42.69

Number of negative sequences: 215834
Number of negative sequences with hit

In [18]:
evaluate(experiment_dirs, 'STREME/streme_evaluator_dummymodel_test.json', 'STREME/streme_evaluator_dummymodel_negative_test.json')

2025-03-21 11:17:22,956 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)
2025-03-21 11:17:24,836 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)


[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak/STREME/s

2025-03-21 11:17:41,443 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:72510161-72511226:0:0-1,065 (2x), chr6:27655855-27656374:0:0-519 (2x)
2025-03-21 11:17:42,034 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr17:72510161-72511226:0:0-1,065 (2x), negative_chr6:27655855-27656374:0:0-519 (2x)
2025-03-21 11:17:44,195 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr22:20748207-20748705:0:0-498 (2x), chr19:36208309-36208837:0:0-528 (2x), chr22:21356041-21356705:0:0-664 (2x), chr6:27100747-27101203:0:0-456 (2x)
2025-03-21 11:17:44,650 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr22:20748207-20748705:0:0-498 (2x), negative_chr19:36208309-36208837:0:0-528 (2x), negative_chr22:21356041-21356705:0:0-664 (2x), negative_chr6:27100747-27101203:0:0-456 (2x)
2025-03-21 11:17:47,148 WARNING: [loadMultiTrainin

[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-21 11:18:01,344 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr6:11537625-11538146:0:0-521 (2x), chr2:175200793-175202077:0:0-1,284 (2x), chrX:37544927-37545500:0:0-573 (2x), chr1:36786772-36787466:0:0-694 (2x), chr1:16173852-16174501:0:0-649 (2x), chr1:234735500-234736178:0:0-678 (2x), chr3:38388039-38388628:0:0-589 (2x), chr19:11071274-11071738:0:0-464 (2x), chr12:124086253-124086801:0:0-548 (2x), chr3:193788616-193789324:0:0-708 (2x), chr6:144536966-144537517:0:0-551 (2x), chr6:159065362-159065766:0:0-404 (2x), chr1:112281896-112282425:0:0-529 (2x), chr7:129074035-129074584:0:0-549 (2x), chr8:98787854-98788312:0:0-458 (2x), chr5:139027661-139028005:0:0-344 (2x), chr1:33116408-33117190:0:0-782 (2x), chr12:108908671-108909083:0:0-412 (2x), chr4:90032070-90032874:0:0-804 (2x), chr7:151328925-151329711:0:0-786 (2x), chr15:41952213-41953334:0:0-1,121 (2x), chr14:100659117-100659576:0:0-459 (2x), chr15:41055323-41056071:0:0-748 (2x), chr1

...

[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsSydhK562CebpbIggrabUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsSydhK562CebpbIggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-21 11:18:38,952 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr9:132597612-132598067:0:0-455 (2x), chr8:142427409-142428437:0:0-1,028 (2x), chr19:19516712-19517481:0:0-769 (2x), chr12:42631191-42631813:0:0-622 (2x), chr7:65447017-65447496:0:0-479 (2x), chr11:72853093-72853622:0:0-529 (2x), chr7:154793980-154794915:0:0-935 (2x), chr5:96270680-96271129:0:0-449 (2x), chr1:156662686-156663301:0:0-615 (2x), chr20:62526719-62527111:0:0-392 (2x), chr3:13036359-13036852:0:0-493 (2x), chr7:43798050-43798484:0:0-434 (2x), chr20:49307836-49308441:0:0-605 (2x), chr16:83986480-83986895:0:0-415 (2x), chr4:6784832-6785381:0:0-549 (2x), chr19:10713038-10713640:0:0-602 (2x), chr22:20849578-20850430:0:0-852 (2x), chr2:106014802-106015752:0:0-950 (2x), chr3:23847521-23848789:0:0-1,268 (2x), chr9:131418639-131419158:0:0-519 (2x), chr19:4867014-4867811:0:0-797 (2x), chr1:17764060-17764595:0:0-535 (2x), chr22:19701615-19702717:0:0-1,102 (2x), chr16:85415487

[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsUwK562CtcfUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsUwK562CtcfUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-21 11:19:02,406 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr4:128702821-128703160:0:0-339 (2x), chr7:100136668-100137233:0:0-565 (2x), chr22:19705322-19706321:0:0-999 (2x), chr1:197871671-197872361:0:0-690 (2x), chr1:43123582-43124140:0:0-558 (2x), chr17:61926089-61927170:0:0-1,081 (2x), chr2:106015376-106015966:0:0-590 (2x), chr22:20067214-20068089:0:0-875 (2x), chr9:37485657-37486115:0:0-458 (2x), chr1:155022843-155023312:0:0-469 (2x), chr11:118868258-118868750:0:0-492 (2x), chr7:135346967-135347443:0:0-476 (2x), chr11:67764043-67764415:0:0-372 (2x), chr7:148725666-148726250:0:0-584 (2x), chr19:12917200-12917697:0:0-497 (2x), chr16:89939464-89940035:0:0-571 (2x), chr13:115079619-115080065:0:0-446 (2x), chr16:81129904-81130331:0:0-427 (2x), chr11:125495140-125495604:0:0-464 (2x), chr1:12123170-12123898:0:0-728 (2x), chr19:59025184-59025713:0:0-529 (2x), chr2:232328405-232328765:0:0-360 (2x), chr5:43120774-43121302:0:0-528 (2x), chr

Total number of test sequences: 216125 | Number of peaks in these sequences: {'bed.tsv': 217928, 'fimo.tsv': 85725, 'mast.tsv': 93968}
Skipped number of test sequences: 83963 | Number of peaks in these sequences: {'bed.tsv': 84108, 'fimo.tsv': 51512, 'mast.tsv': 55107}
Skipped 7/40 experiments: ['/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/test_20250320/wgEncodeAwgTfbsBroadK562CtcfUniPk.narrowPeak', '/home/matthis/PhD/mn

---

Distribution of test sequence number and peaks per experiment, seems to be quite unevenly distributed.

In [19]:
n_test_seqs = []
n_peaks = {}
i_skipped = set()

evaluator_path = "STREME/streme_evaluator_dummymodel_test.json"
neg_evaluator_path = "STREME/streme_evaluator_dummymodel_negative_test.json"

for i, ed in enumerate(experiment_dirs):
    assert (ed / 'test_sequences_0.json').exists()
    testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
    skipping = (not (ed / evaluator_path).exists()) or (neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists())

    n_test_seqs.append( sum([len(g) for g in testdata]) )

    for g in testdata:
        for s in g:
            assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
            for e in s.genomic_elements:
                peaksrc = e.source
                if peaksrc not in n_peaks:
                    n_peaks[peaksrc] = [0]*i # in the (unlikely) case that experiments [0, i) did not have this type
                assert len(n_peaks[peaksrc]) >= i, f"{peaksrc}, {i}, {n_peaks}" # should not fail
                if len(n_peaks[peaksrc]) == i:
                    n_peaks[peaksrc].append(0)

                n_peaks[peaksrc][i] += 1

    if not (ed / evaluator_path).exists():
        i_skipped.add(i)
        continue

    if neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists():
        i_skipped.add(i)
        continue

In [20]:
import importlib
importlib.reload(plotting)

<module 'modules.plotting' from '/home/matthis/PhD/genomegraph/learn_specific_profiles/modules/plotting.py'>

In [21]:
fig = plotting.ownPlotlyHist(lists={'all experiments': n_test_seqs, 'skipped experiments': [n_test_seqs[i] for i in i_skipped]}, binSize=500)
fig.update_layout(title_text="Experiments and their number of test sequences", xaxis_title="Number of test sequences", yaxis_title="No. exp. with this no. test sequences")
fig.show()

In [22]:
for peaksrc in n_peaks:
    fig = plotting.ownPlotlyHist(lists={'all experiments': n_peaks[peaksrc], 'skipped experiments': [n_peaks[peaksrc][i] for i in i_skipped]}, binSize=200)
    fig.update_layout(title_text=f"Experiments and their number of peaks in test sequences from {peaksrc}", xaxis_title="Number of peaks", yaxis_title="No. exp. with this no. peaks")
    fig.show()
# fig = plotting.ownPlotlyHist(lists=n_peaks, binSize=200)
# fig.update_layout(title_text="Experiments and their number of peaks in test sequences", xaxis_title="Number of peaks", yaxis_title="No. exp. with this no. peaks")
# fig.show()

---

Distribution of average peak content per experiment (percent of sequences with peaks, average number of peaks per sequence)

In [23]:
peaksrcs = set()
for ed in experiment_dirs:
    assert (ed / 'test_sequences_0.json').exists()
    testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
    for g in testdata:
        for s in g:
            assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
            for e in s.genomic_elements:
                peaksrcs.add(e.source)


data = {peaksrc: {'n_test_seqs': [], 'n_test_seqs_with_peaks': [], 'n_peaks': {}} for peaksrc in peaksrcs}
for peaksrc in peaksrcs:
    for ed in experiment_dirs:
        assert (ed / 'test_sequences_0.json').exists()
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))

        data[peaksrc]['n_test_seqs'].append( sum([len(g) for g in testdata]) )

        n_seqs_with_peaks = 0
        n_peaks = 0
        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                hasPeak = False
                for e in s.genomic_elements:
                    if e.source == peaksrc:
                        hasPeak = True
                        n_peaks += 1
                if hasPeak:
                    n_seqs_with_peaks += 1

        data[peaksrc]['n_test_seqs_with_peaks'].append(n_seqs_with_peaks)
        data[peaksrc]['n_peaks'][ed] = n_peaks

In [24]:
for peaksrc in peaksrcs:
    data[peaksrc]['perc_seq_with_peaks'] = [100*n/nseqs for n, nseqs in zip(data[peaksrc]['n_test_seqs_with_peaks'], data[peaksrc]['n_test_seqs'])]
    data[peaksrc]['avg_num_peaks'] = [n/nseqs for n, nseqs in zip(data[peaksrc]['n_peaks'].values(), data[peaksrc]['n_test_seqs'])]

In [25]:
fig = plotting.ownPlotlyHist(lists={peaksrc: data[peaksrc]['perc_seq_with_peaks'] for peaksrc in peaksrcs}, binSize=10)
fig.update_layout(title_text="Experiments and their percentage of sequences with peaks", xaxis_title="Percentage of sequences with peaks", yaxis_title="No. exp. with this percentage")
fig.show()

In [26]:
fig = plotting.ownPlotlyHist(lists={peaksrc: data[peaksrc]['avg_num_peaks'] for peaksrc in peaksrcs}, binSize=0.1)
fig.update_layout(title_text="Experiments and their average number of peaks per sequence", xaxis_title="Average number of peaks per sequence", yaxis_title="No. exp. with this average")
fig.show()

Peak count per sequence (not per experiment)

In [27]:
data = {peaksrc: [] for peaksrc in peaksrcs}
for peaksrc in peaksrcs:
    for ed in experiment_dirs:
        assert (ed / 'test_sequences_0.json').exists()
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))

        for g in testdata:
            for s in g:
                n_peaks = 0
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    if e.source == peaksrc:
                        n_peaks += 1

                data[peaksrc].append(n_peaks)

In [28]:
fig = plotting.ownPlotlyHist(lists=data, binSize=1, rel=True, xlim=(0, 6))
fig.update_layout(title_text="Number of peaks in test sequences", xaxis_title="Number of peaks", yaxis_title="No. seq. with this no. peaks")
fig.show()